In [3]:
!pip install yfinance pandas numpy ta scikit-learn

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=fe0f18082324c567276254bf3364789a8c28b5a25324b94e06b03acb858ed0ca
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


# Setup & Imports

In [4]:
# ---- ML Simple Long/Flat Strategy (SPY/VFV) ----
# File: ML_Simple_LongFlat_SPY.ipynb

import yfinance as yf
import pandas as pd
import numpy as np

from ta.momentum import RSIIndicator
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)


# Download Historical Data

In [5]:
# Choose your asset: SPY or VFV.TO
ticker = "^GSPC"     # change to "VFV.TO" for the Canadian S&P 500 ETF

data = yf.download(ticker, period="5y")

# Fix multi-index columns
if isinstance(data.columns, pd.MultiIndex):
    data.columns = [c[0] for c in data.columns]

data.head()


/tmp/ipykernel_12647/3638004128.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="5y")
[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume
Date,,,,,
2021-03-31,3972.889893,3994.409912,3966.979980,3967.250000,4578050000
2021-04-01,4019.870117,4020.629883,3992.780029,3992.780029,4162130000
2021-04-05,4077.909912,4083.419922,4034.439941,4034.439941,4005030000
2021-04-06,4073.939941,4086.229980,4068.139893,4075.570068,4081270000
2021-04-07,4079.949951,4083.129883,4068.310059,4074.290039,4120810000


In [6]:
last_date = data.index[-1]
print(f"Latest date: {last_date}")
print(data.loc[last_date])




Latest date: 2026-03-30 00:00:00
Close     6.343720e+03
High      6.427310e+03
Low       6.316910e+03
Open      6.403370e+03
Volume    3.264973e+09
Name: 2026-03-30 00:00:00, dtype: float64


# Feature Engineering

In [7]:
# 1D, 5D, 10D returns
data["ret_1d"] = data["Close"].pct_change(1)
data["ret_5d"] = data["Close"].pct_change(5)
data["ret_10d"] = data["Close"].pct_change(10)

# RSI
close_series = data["Close"]
rsi = RSIIndicator(close_series, window=14)
data["rsi_14"] = rsi.rsi()

# 10D volatility
data["vol_10d"] = data["ret_1d"].rolling(10).std()

# Forward return + label
data["ret_fwd_1d"] = data["Close"].pct_change().shift(-1)
data["y"] = (data["ret_fwd_1d"] > 0).astype(int)


data.tail()


,Close,High,Low,Open,Volume,ret_1d,ret_5d,ret_10d,rsi_14,vol_10d,ret_fwd_1d,y
Date,,,,,,,,,,,,
2026-03-24,6556.370117,6595.750000,6525.109863,6552.089844,5278580000,-0.003743,-0.023782,-0.033195,35.857450,0.009629,0.005419,1
2026-03-25,6591.899902,6633.939941,6568.410156,6598.350098,4936720000,0.005419,-0.004951,-0.027141,39.231909,0.010005,-0.017406,0
2026-03-26,6477.160156,6573.220215,6473.790039,6555.859863,4845560000,-0.017406,-0.019576,-0.029293,33.164097,0.010326,-0.016722,0
2026-03-27,6368.850098,6453.890137,6356.080078,6453.890137,5303490000,-0.016722,-0.021153,-0.039706,28.658201,0.011200,-0.003946,0
2026-03-30,6343.720215,6427.310059,6316.910156,6403.370117,3264973000,-0.003946,-0.036055,-0.053088,27.717244,0.010054,NaN,0


In [8]:
print(data.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'ret_1d', 'ret_5d', 'ret_10d',
       'rsi_14', 'vol_10d', 'ret_fwd_1d', 'y'],
      dtype='object')


# Prepare Training Data

In [9]:
# Make sure these are the features we expect
features = ["ret_1d", "ret_5d", "ret_10d", "rsi_14", "vol_10d"]

# This will raise a clear error if something is missing
missing_cols = [c for c in features + ["y"] if c not in data.columns]
if missing_cols:
    raise ValueError(f"These columns are missing from 'data': {missing_cols}. "
                     "Run the feature-engineering cell first.")

# Build a clean dataframe with only the columns we need, then drop NaNs
df = data[features + ["y"]].dropna().copy()

X = df[features]
y = df["y"]

from sklearn.model_selection import train_test_split

# No shuffle for time-series-ish split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

X_train.tail(), X_test.head()



(              ret_1d    ret_5d   ret_10d     rsi_14   vol_10d
 Date                                                         
 2025-03-26 -0.011157  0.006504  0.020163  45.457176  0.012100
 2025-03-27 -0.003307  0.005372  0.031113  44.258871  0.010958
 2025-03-28 -0.019737 -0.015283 -0.010286  37.864501  0.011085
 2025-03-31  0.005539 -0.026999 -0.011149  40.414686  0.011024
 2025-04-01  0.003781 -0.024855  0.003279  42.169426  0.010564,
               ret_1d    ret_5d   ret_10d     rsi_14   vol_10d
 Date                                                         
 2025-04-02  0.006728 -0.007218 -0.000761  45.269631  0.010190
 2025-04-03 -0.048396 -0.052130 -0.047038  31.923661  0.018426
 2025-04-04 -0.059750 -0.090820 -0.104715  23.250965  0.025154
 2025-04-07 -0.002331 -0.097936 -0.122291  23.004037  0.023383
 2025-04-08 -0.015701 -0.115443 -0.137429  21.362556  0.022843)

# Train Logistic Regression Model

In [10]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))


Train accuracy: 0.527693856998993
Test accuracy: 0.5542168674698795


# Generate Today’s Trade Signal

In [11]:
# Use the most recent row to generate today's signal
latest_features = X.iloc[[-1]]
p_up = model.predict_proba(latest_features)[0, 1]

print(f"Predicted probability the market is UP tomorrow: {p_up:.3f}")

signal = "LONG" if p_up > 0.55 else "CASH/FLAT"
print(f"Trade Signal Today: {signal}")


Predicted probability the market is UP tomorrow: 0.520
Trade Signal Today: CASH/FLAT


# Final Output (For DALIS Trade Log)

In [12]:
from datetime import datetime

print("----- DALIS ML Portfolio Trade Signal -----")
print(f"Date: {datetime.today().strftime('%Y-%m-%d')}")
print(f"Ticker: {ticker}")
print(f"Model: Logistic Regression (Daily Long/Flat)")
print(f"Probability Market Up Tomorrow: {p_up:.4f}")
print(f"Action: {signal}")
print("--------------------------------------------")


----- DALIS ML Portfolio Trade Signal -----
Date: 2026-03-31
Ticker: ^GSPC
Model: Logistic Regression (Daily Long/Flat)
Probability Market Up Tomorrow: 0.5200
Action: CASH/FLAT
--------------------------------------------
